In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
np.random.seed(42)
random.seed(42)

N = 1000  # number of employees
departments = ['Engineering', 'Consulting', 'Sales', 'HR', 'Finance', 'Operations']
roles = ['Analyst', 'Senior Analyst', 'Consultant', 'Manager', 'Senior Manager']
job_levels = ['L1', 'L2', 'L3', 'L4', 'L5']

# ---------- 1. EMPLOYEE MASTER ----------
employee_ids = [f"EMP{1000+i}" for i in range(N)]
hire_dates = [fake.date_between(start_date='-8y', end_date='-30d') for _ in range(N)]
tenure_years = [(datetime.now().date() - d).days / 365 for d in hire_dates]

departments_col = np.random.choice(departments, N)
salary = np.random.normal(900000, 300000, N).clip(300000, 3000000).astype(int)
performance = np.random.choice(['Low', 'Average', 'High'], N, p=[0.15, 0.6, 0.25])

# Bench days will influence attrition — generate bench first, then attrition based on it
bench_days = np.random.exponential(scale=25, size=N).clip(0, 250).astype(int)

# attrition probability increases with bench days and low performance
attr_prob = 0.05 + (bench_days / 250) * 0.35 + (performance == 'Low') * 0.15
attr_prob = np.clip(attr_prob, 0, 0.9)
attrition_flag = np.random.binomial(1, attr_prob)

exit_dates = [
    fake.date_between(start_date=hire_dates[i], end_date='today') if attrition_flag[i] else None
    for i in range(N)
]

employee_master = pd.DataFrame({
    'employee_id': employee_ids,
    'name': [fake.name() for _ in range(N)],
    'department': departments_col,
    'role': np.random.choice(roles, N),
    'job_level': np.random.choice(job_levels, N),
    'hire_date': hire_dates,
    'tenure_years': np.round(tenure_years, 2),
    'salary': salary,
    'performance_rating': performance,
    'manager_id': np.random.choice(employee_ids, N),
    'attrition_flag': attrition_flag,
    'exit_date': exit_dates
})

# ---------- 2. BENCH & PROJECT ALLOCATION ----------
project_ids = [f"PRJ{100+i}" for i in range(50)]
utilization_status = []
project_id_col = []

for bd in bench_days:
    if bd > 15:
        utilization_status.append('Bench')
        project_id_col.append(None)
    else:
        utilization_status.append(random.choice(['Billed', 'Training']))
        project_id_col.append(random.choice(project_ids))

bench_allocation = pd.DataFrame({
    'employee_id': employee_ids,
    'project_id': project_id_col,
    'allocation_start_date': hire_dates,
    'bench_days': bench_days,
    'utilization_status': utilization_status
})

# ---------- 3. FINANCE / COST ----------
daily_rate = salary / 260  # working days/year approx
bench_cost = (bench_days * daily_rate).astype(int)

seniority_multiplier = np.select(
    [performance == 'High', performance == 'Average', performance == 'Low'],
    [1.8, 1.3, 0.8]
)
replacement_cost = (salary * seniority_multiplier).astype(int)

finance_cost = pd.DataFrame({
    'employee_id': employee_ids,
    'monthly_ctc': (salary / 12).astype(int),
    'bench_cost_incurred': bench_cost,
    'replacement_cost_estimate': replacement_cost
})

# ---------- SAVE ----------
employee_master.to_csv('../data/employee_master.csv', index=False)
bench_allocation.to_csv('../data/bench_allocation.csv', index=False)
finance_cost.to_csv('../data/finance_cost.csv', index=False)

print("Done. Rows generated:", len(employee_master))
employee_master.head()

Done. Rows generated: 1000


,employee_id,name,department,role,job_level,hire_date,tenure_years,salary,performance_rating,manager_id,attrition_flag,exit_date
0,EMP1000,Tanya Armstrong,HR,Consultant,L4,2025-02-05,1.48,704807,Low,EMP1296,0,None
1,EMP1001,Sara Melendez,Finance,Consultant,L4,2022-02-10,4.47,753862,Average,EMP1509,0,None
2,EMP1002,Kristen Powell,Sales,Analyst,L5,2023-11-21,2.69,722281,High,EMP1714,0,None
3,EMP1003,Alexander Weaver,Finance,Senior Analyst,L2,2024-01-04,2.57,640802,Average,EMP1983,0,None
4,EMP1004,Brian Walker,Finance,Consultant,L2,2022-10-01,3.83,914556,High,EMP1965,0,None
